# Actividad 3. Explorando relaciones entre categorías: perfiles sociodemográficos y estado glucémico en NHANES

**Técnicas de Análisis No Supervisado** · Especialización en Análisis Estadístico para Ciencia de Datos · Universidad de La Salle

## 1. Configuración

In [1]:
import os
import sys

# Configuración para Google Colab: clona el repositorio, instala dependencias y descarga los datos.
# En una ejecución local esta celda no hace nada.
if "google.colab" in sys.modules:
    if not os.path.exists("/content/nhanes-no-supervisado"):
        !git clone https://github.com/Juanhv24/nhanes-no-supervisado.git /content/nhanes-no-supervisado
    %cd /content/nhanes-no-supervisado
    !pip install -q -e .
    !python scripts/descargar_datos.py
    # Garantiza que el kernel encuentre el paquete aunque no se reinicie la sesión
    sys.path.insert(0, "/content/nhanes-no-supervisado/src")

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import prince
from scipy.stats import chi2_contingency

from nhanes_no_supervisado.config import SEMILLA, ruta_figuras
from nhanes_no_supervisado.datos import cargar_crudos

RUTA_FIGURAS = ruta_figuras(3)

## 2. Problema y contexto

### 2.1 Pregunta guía y relevancia

Los determinantes sociales de la salud son las circunstancias en que las personas nacen, crecen, viven, trabajan y envejecen, y la Organización Mundial de la Salud les dedicó una comisión específica, cuyo informe final concluyó que las desigualdades en salud tienen un origen marcadamente social (Comisión sobre Determinantes Sociales de la Salud [CSDH], 2008). En el caso de la diabetes, la American Diabetes Association reconoce que factores como el nivel socioeconómico, el entorno del vecindario y la vivienda, el entorno alimentario, incluida la inseguridad alimentaria, y el acceso a la atención en salud se asocian con el riesgo de diabetes y con sus resultados clínicos (Hill-Briggs et al., 2021). De forma coherente, una revisión de alcance en población con diabetes mellitus tipo 2 identificó determinantes estructurales, como la seguridad financiera, e intermediarios, como la afiliación médica, el acceso a los servicios de salud y las condiciones de la vivienda (Heredia-Morales et al., 2023).

En consecuencia, la atención de la diabetes no puede limitarse a los aspectos puramente biomédicos, puesto que las condiciones de vida de cada persona influyen en su posibilidad real de acceder a un tratamiento asequible y equitativo (Organización Panamericana de la Salud [OPS], s.f.). Por ello, identificar qué perfiles sociodemográficos y de estilo de vida concentran los estados glucémicos desfavorables permite orientar la detección temprana y la prevención hacia los grupos en mayor desventaja. Con este propósito, además de las características demográficas, el análisis incorpora dos de los determinantes señalados por la literatura que NHANES mide directamente: la seguridad alimentaria y la cobertura de seguro médico. A partir de lo anterior, la pregunta guía de esta actividad es: *¿Qué perfiles sociodemográficos y de estilo de vida se asocian con el estado glucémico de los adultos?*

Dado que tanto el estado glucémico como las características sociodemográficas son variables categóricas, el análisis de correspondencias resulta adecuado para responderla: el análisis simple permite estudiar la asociación entre dos de ellas a partir de una tabla de contingencia, mientras que el múltiple permite explorar de forma conjunta las relaciones entre varias variables y representarlas en un mismo mapa perceptual.

### 2.2 Base de datos

La Encuesta Nacional de Examen de Salud y Nutrición (NHANES, por sus siglas en inglés) es un programa del Centro Nacional de Estadísticas de Salud (NCHS) de los Centros para el Control y la Prevención de Enfermedades (CDC) de Estados Unidos, diseñado para evaluar el estado de salud y nutrición de su población (National Center for Health Statistics [NCHS], s.f.). La encuesta combina entrevistas en el hogar con exámenes físicos y pruebas de laboratorio, de modo que reúne en un mismo participante características sociodemográficas, hábitos y mediciones clínicas; además, se publica en ciclos de dos años divididos en módulos que se vinculan mediante un identificador único por participante (`SEQN`) (NCHS, s.f.).

En este trabajo se emplea el ciclo 2017–2018, el mismo de la actividad anterior, y se usan seis de sus módulos: demografía (edad, sexo, nivel educativo, raza y origen étnico, y razón entre el ingreso familiar y la línea de pobreza), hemoglobina glicada, a partir de la cual se define el estado glucémico, medidas corporales, para la obesidad abdominal, tabaquismo, seguridad alimentaria y seguro médico.

## 3. Población de estudio

### 3.1 Carga y unión de los módulos

In [3]:
df = cargar_crudos()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9254 entries, 93703 to 102956
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   edad                   9254 non-null   float64
 1   sexo                   9254 non-null   float64
 2   embarazo               1110 non-null   float64
 3   educacion              5569 non-null   float64
 4   raza_etnia             9254 non-null   float64
 5   ingreso_pobreza        8023 non-null   float64
 6   cintura_cm             7601 non-null   float64
 7   imc                    8005 non-null   float64
 8   hba1c_pct              6045 non-null   float64
 9   glucosa_mgdl           2891 non-null   float64
 10  trigliceridos_mgdl     2834 non-null   float64
 11  hdl_mgdl               6738 non-null   float64
 12  diabetes_dx            8897 non-null   float64
 13  fumo_100               5856 non-null   float64
 14  fuma_actual            2359 non-null   float64
 15  seguridad

### 3.2 Selección de la población

In [4]:
# Adultos de 20 años o más, sin embarazo confirmado y con HbA1c disponible
es_adulto = df["edad"] >= 20
es_embarazada = df["embarazo"] == 1

df_correspondencias = df[es_adulto & ~es_embarazada].drop(columns="embarazo")
df_correspondencias = df_correspondencias.dropna(subset=["hba1c_pct"]).copy()

print(f"Adultos sin embarazo: {(es_adulto & ~es_embarazada).sum():,}")
print(f"Con HbA1c disponible: {len(df_correspondencias):,}")

Adultos sin embarazo: 5,514
Con HbA1c disponible: 4,971


La población de estudio se restringe a adultos de 20 años o más, que es el corte habitual en los estudios sobre salud cardiometabólica basados en NHANES (Hirode & Wong, 2020), y se excluyen las participantes con embarazo confirmado, puesto que la gestación modifica de forma marcada el perfil metabólico (Wang et al., 2016). Además, se conservan solo los participantes con hemoglobina glicada (HbA1c) disponible, ya que a partir de ella se define el estado glucémico. A diferencia de la glucosa en ayunas, la HbA1c no requiere ayuno y es uno de los criterios diagnósticos de la diabetes y la prediabetes (American Diabetes Association Professional Practice Committee [ADA], 2025), de modo que el análisis no se limita a la submuestra matutina y conserva un número considerablemente mayor de participantes.

## Referencias

American Diabetes Association Professional Practice Committee. (2025). 2. Diagnosis and classification of diabetes: Standards of Care in Diabetes—2025. *Diabetes Care, 48*(Supl. 1), S27–S49. https://doi.org/10.2337/dc25-S002

Comisión sobre Determinantes Sociales de la Salud. (2008). *Subsanar las desigualdades en una generación: Alcanzar la equidad sanitaria actuando sobre los determinantes sociales de la salud*. Organización Mundial de la Salud.

Heredia-Morales, M., Ramírez-Jaime, L. E., & Miranda-Cota, G. A. (2023). Determinantes sociales de salud y diabetes mellitus tipo 2: Una revisión de alcance. En *Salud y alimentación desde una perspectiva sociocultural* (pp. 204–220). Astra Ediciones. https://doi.org/10.61728/AE24140119

Hill-Briggs, F., Adler, N. E., Berkowitz, S. A., Chin, M. H., Gary-Webb, T. L., Navas-Acien, A., Thornton, P. L., & Haire-Joshu, D. (2021). Social determinants of health and diabetes: A scientific review. *Diabetes Care, 44*(1), 258–279. https://doi.org/10.2337/dci20-0053

Hirode, G., & Wong, R. J. (2020). Trends in the prevalence of metabolic syndrome in the United States, 2011–2016. *JAMA, 323*(24), 2526–2528. https://doi.org/10.1001/jama.2020.4501

National Center for Health Statistics. (s.f.). *National Health and Nutrition Examination Survey*. Centers for Disease Control and Prevention. https://www.cdc.gov/nchs/nhanes/

Organización Panamericana de la Salud. (s.f.). *Diabetes*. https://www.paho.org/es/temas/diabetes

Wang, Q., Würtz, P., Auro, K., Mäkinen, V.-P., Kangas, A. J., Soininen, P., Tiainen, M., Tynkkynen, T., Jokelainen, J., Santalahti, K., Salmi, M., Blankenberg, S., Zeller, T., Viikari, J., Kähönen, M., Lehtimäki, T., Salomaa, V., Perola, M., Jalkanen, S., . . . Ala-Korpela, M. (2016). Metabolic profiling of pregnancy: Cross-sectional and longitudinal evidence. *BMC Medicine, 14*, Artículo 205. https://doi.org/10.1186/s12916-016-0733-0